# Uniform B-Splines

Parametric piece-wise curve, determined by set of control points and associated basis polinomial functions of degree $d$:

$$S^{(d)}(t): \mathbb{R} \to \mathbb{R}^3 \tag{curve}$$

Parametric basis functions are the polynomials of degree $d$:

$$B^{(d)}(t): \mathbb{R} \to \mathbb{R} \tag{basis}$$

In general case:

- $t$ runs along path infinitely
- The path is subdivided into knots $t_j$
- A $j$'th segment of the curve corresponds to a span of path with $t \in [t_{i}, t_{i+1})$
- A $j$'th segment is affected by $d+1$ nearby control points and assosiated basis functions
- Each basis function is associated with a control point and determines its weight

The _nearby_ is arbitrary mapping between knots and controls.

In uniform (unit-spaced) case:

- $0 \lt t_{local} \lt 1$ for each segment
- Evaluation of (sum of) all B-functions are all the same for each segment.
- Each segment is determined by a set of control points and (sum of) basis functions


# Generic matrix form

$$
B^{(d)}(t) = [1, t^1, \dots, t^d] \; M^{(d)}
$$

$$
S_j^{(d)}(t) = [1, t^1, \dots, t^d] \; M^{(d)} \; C^{(d)}_j
$$

The $[1, t^1, \dots, t^d]$ is a power basis simply to isolate dependency on t

The $B^{(d)}$ is a row vector of $d+1$ basis functions affecting curve at a point. They are the same for every segment.

The $C^{(d)}$ is not a real matrix, it's a column-vector of $d+1$ control points in $\mathbb{R}^2$ or $\mathbb{R}^3$ (or anything else in general), affecting curve at a particular segment $j$


## The B-matrices

$$
M^{(1)} =
\begin{pmatrix}
1 & 0 \\
-1 & 1
\end{pmatrix}
\tag{linear}
$$

The linear case is the same as LERP: $[1, t] M C = [1, t] \big[ c_1, c_2 - c_1\big]^T = c_1 + (c_2 - c_1) t = c_1 (1 - t) + c_2 (t)$

$$
M^{(2)} = \frac{1}{2}
\begin{pmatrix}
1 & 1 & 0 \\
-2 & 2 & 0 \\
1 & -2 & 1
\end{pmatrix}
\tag{quadratic}
$$

$$
M^{(3)} = \frac{1}{6}
\begin{pmatrix}
1 & 4 & 1 & 0 \\
-3 & 0 & 3 & 0 \\
3 & -6 & 3 & 0 \\
-1 & 3 & -3 & 1
\end{pmatrix}
\tag{cubic}
$$


In [ ]:
from typing import NamedTuple, Self

import numpy as np
from numpy.typing import NDArray

Vec = np.ndarray[tuple[int], np.dtype[np.floating]]  # 2D or 3D
nparr = lambda *args: np.array(args)
nparrg = lambda iter: np.array(tuple(iter))

In [ ]:
def powers(d: int, t: float) -> NDArray:
    return np.power(t, np.arange(d + 1))


def powersnp(d: int, t: NDArray) -> NDArray:
    return np.power(t[..., None], np.arange(d + 1))

In [ ]:
M_ = {
    1: np.array([[1, 0], [-1, 1]]),
    2: np.array([[1, 1, 0], [-2, 2, 0], [1, -2, 1]]) / 2,
    3: np.array([[1, 4, 1, 0], [-3, 0, 3, 0], [3, -6, 3, 0], [-1, 3, -3, 1]]) / 6,
}


# FIXME: make same d
Mdt_ = {
    1: np.array([[-1, 1]]),
    2: np.array([[-1, 1, 0], [1, -2, 1]]),
    3: np.array([[-1, 0, 1, 0], [2, -4, 2, 0], [-1, 3, -3, 1]]) / 2,
}

In [ ]:
def controls_iter(d: int, controls: NDArray):
    """Iterate over all possible segments and yield their control points
    (d+1 points for each segment)
    """
    for i in range(len(controls) - d):
        yield controls[i : i + d + 1]

In [ ]:
JIGG = nparr(0.5, 0.5, 0.25)


def random_points(n: int):
    xx = np.linspace(1.0, 0.0, n)
    yy = np.linspace(0.0, 1.0, n)
    zz = np.zeros(n)

    points = np.stack((xx, yy, zz)).T
    points[1:-1] += (np.random.random((n - 2, 3)) - 0.5) * 2 * JIGG
    return points

#### The Basis


In [ ]:
def basis(d: int, t: float) -> NDArray:
    return powers(d, t) @ M_[d]


def basisnp(d: int, tspace: NDArray) -> NDArray:
    return powersnp(d, tspace) @ M_[d]

#### The curve


In [ ]:
def segm_point(d: int, controls: NDArray, t: float):
    """A point on a curve segment on the controls
    - t in [0, 1]
    - the controls should be of d+1
    """
    return basis(d, t) @ controls


def segm_curve(d: int, controls: NDArray, tspace: NDArray):
    """All points on a curve segment on the controls
    - for all t in local tspace 0 <= t <= 1
    - the controls should be of d
    """
    assert len(controls) == d + 1
    B = basisnp(d, tspace)
    return B @ controls


def mega_curve(d: int, controls: NDArray, tspace: NDArray):
    """All points of curve concatenated"""
    assert len(controls) > d + 1
    B = basisnp(d, tspace)
    return np.concat([B @ ctrl for ctrl in controls_iter(d, controls)])

In [ ]:
def segm_flow(d: int, controls: NDArray, tspace: NDArray):
    """A velocity on a curve segment at t
    - t in [0, 1]
    - the controls should be of d
    """
    Bdt = powersnp(d - 1, tspace) @ Mdt_[d]
    return Bdt @ controls


def mega_flow(d: int, controls: NDArray, tspace: NDArray):
    assert len(controls) > d + 1
    Bdt = powersnp(d - 1, tspace) @ Mdt_[d]
    return np.concat([Bdt @ ctrl for ctrl in controls_iter(d - 1, controls)])

---


In [ ]:
from ipywidgets import widgets as wg
import plotly.graph_objects as go
from widgets import Curve3DView, Curve2DView

### Plotting Basis


In [ ]:
D = 3
tspace = np.linspace(0, 1, 16)
bb = basisnp(D, tspace)

In [ ]:
go.Figure(
    data=[go.Scatter(x=tspace, y=b, mode="lines") for b in bb.T],
    layout=dict(title=f"Basis functions d={D}"),
)

### Plotting curves


In [ ]:
D = 2
controls = random_points(D + 1)
controls2d = controls[:, :2]

In [ ]:
wg.HBox([
    Curve3DView(
        f"B({D})-Spline ",
        points=controls,
        curve_fn=lambda tt: segm_curve(D, controls, tt),
        flow_fn=lambda tt: segm_flow(D, controls, tt),
    ),
    Curve2DView(
        f"B({D})-Spline flat",
        points=controls2d,
        curve_fn=lambda tt: segm_curve(D, controls2d, tt),
        flow_fn=lambda tt: segm_flow(D, controls2d, tt),
    ),
])

# Quadratic Bezier

A segment of B(2)-spline on $c_{j-1}, c_{j}, c_{j+1}$ is a quadratic bezier on $c_1 = \frac{c_{j-1} + c_{j}}{2}, c_2 = c_j, c_3 = \frac{c_{j+1} + c_{j}}{2}$
With $c_2 = c_j$ at it's zenith.

$$
S(t) = [1, t, t^2]
\begin{pmatrix}
1 & 0 & 0 \\
-2 & 2 & 0 \\
1 & -2 & 1 \\
\end{pmatrix}
\begin{bmatrix}
c_1 \\
c_2 \\
c_3 \\
\end{bmatrix}
$$

$$
\frac{dS}{dt}(t) = [1, t, t^2]
\begin{pmatrix}
-2 & 2 & 0 \\
2 & -4 & 2 \\
0 & 0 & 0 \\
\end{pmatrix}
\begin{bmatrix}
c_1 \\
c_2 \\
c_3 \\
\end{bmatrix}
$$


In [ ]:
B2 = np.array([[1, 0, 0], [-2, 2, 0], [1, -2, 1]])
B2D = np.array([[-2, 2, 0], [2, -4, 2], [0, 0, 0]])


def b2zier_point(controls: NDArray, t: float):
    """A point on a bezier curve between the controls
    - t in [0, 1]
    - the controls should be of 3
    """
    assert len(controls) == 3
    M = powers(2, t) @ B2
    return M @ controls


def b2zier_curve(controls: NDArray, tspace: NDArray):
    """All points on a bezier curve between the controls
    - for all t in local tspace 0 <= t <= 1
    - the controls should be of 3
    """
    assert len(controls) == 3
    M = powersnp(2, tspace) @ B2
    return M @ controls


def b2zier_flow(controls: NDArray, tspace: NDArray):
    assert len(controls) == 3
    M = powersnp(2, tspace) @ B2D
    return M @ controls

In [ ]:
controls = random_points(3)
controls2d = controls[:, :2]

In [ ]:
wg.HBox([
    Curve3DView(
        "B(2)ezier segment",
        points=controls,
        curve_fn=lambda tt: b2zier_curve(controls, tt),
        flow_fn=lambda tt: b2zier_flow(controls, tt),
    ),
    Curve2DView(
        "B(2)ezier segment flat",
        points=controls2d,
        curve_fn=lambda tt: b2zier_curve(controls2d, tt),
        flow_fn=lambda tt: b2zier_flow(controls2d, tt),
    ),
])

# Quadratic Beziel

Centering around apex point and shifting $t$.

- $c_2 \to 0$
- $t \in [0, 1] \to t' \in [-0.5, +0.5]$
- $c_1 \to v_1 = c_1 - c_2$
- $c_3 \to v_3 = c_3 - c_2$
- $v_a = v_3 + v_1$ — parabolic axial vector
- $v_t = v_3 - v_1$ — parabolic tangential vector
- $c_a = c_2 + .25 v_a = S(0)$ — apex

Within basis axes $v_t, v_a$ the curve is just a parabola $y = x^2$, skewed.

$$
S(t') - c_a=
\big[1, t', t'^2 \big]
\begin{bmatrix}
0 \\
v_t \\
v_a \\
\end{bmatrix}
$$

$$
\frac{dS}{dt}(t') =
\big[1, t', t'^2 \big]
\begin{bmatrix}
v_t \\
2 v_a \\
0 \\
\end{bmatrix}
$$

Tho, the transform from/into the $v_t, v_a$ does not preserve perpendicularity and distances.


In [1]:
class B2zielParams(NamedTuple):
    """Params of a bezier segment on points c_1 c_2 c_3
    With basis at apex point and shifted t
    """

    c2: Vec  # zenith
    v1: Vec  # c2 > c1
    v3: Vec  # c2 > c3

    ca: Vec  # apex
    va: Vec  # parabolix axis
    vt: Vec  # parabolic tangent

    @classmethod
    def Init(cls, c1: Vec, c2: Vec, c3: Vec) -> Self:
        v1 = c1 - c2
        v3 = c3 - c2
        va = v3 + v1
        vt = v3 - v1
        ca = c2 + 0.25 * va
        return cls(c2, v1, v3, ca, va, vt)

    @property
    def M(self):
        return nparr(np.zeros(self.c2.shape), self.vt, self.va)

    @property
    def Mdt(self):
        return nparr(self.vt, 2 * self.va, np.zeros(self.c2.shape))


def b2ziel_curve(params: B2zielParams, tspace: NDArray):
    return powersnp(2, tspace - 0.5) @ params.M + params.ca


def b2ziel_flow(params: B2zielParams, tspace: NDArray):
    return powersnp(2, tspace - 0.5) @ params.Mdt

NameError: name 'NamedTuple' is not defined

In [ ]:
controls = random_points(3)
controls2d = controls[:, :2]
params = B2zielParams.Init(*controls)
params2d = B2zielParams.Init(*controls2d)

In [ ]:
wg.HBox([
    Curve3DView(
        "B(2)eziel segment",
        points=controls,
        curve_fn=lambda tt: b2ziel_curve(params, tt),
        flow_fn=lambda tt: b2ziel_flow(params, tt),
    ),
    Curve2DView(
        "B(2)eziel segment flat",
        points=controls2d,
        curve_fn=lambda tt: b2ziel_curve(params2d, tt),
        flow_fn=lambda tt: b2ziel_flow(params2d, tt),
    ),
])